# 量子文本情感分类模型

本教程展示一个小型量子情感分类模型：把三词英文句子表示成量子电路，通过训练电路中的旋转角参数，让电路学会区分积极和消极情感。

整个流程分成六步：

1. 准备由主语、动词、宾语组成的简单句子
2. 把每个词编码成量子旋转门
3. 用 CNOT 门把主语、动词、宾语组合成一个句子电路
4. 从电路测量结果中计算情感概率
5. 直接优化电路参数
6. 保存训练结果，并在重新加载后进行推理

## 环境安装

首次运行前可以安装这些依赖：

In [ ]:
!pip install numpy scipy matplotlib qiskit qiskit-aer

本教程不需要PyTorch。训练使用的是 `scipy.optimize`，量子测量采样和电路绘图使用的是 Qiskit。

## 1. 导入环境

这一节导入整个实验需要的工具：

- `numpy`: 保存和更新电路参数
- `matplotlib`: 绘制训练 loss 曲线
- `scipy.optimize.minimize`: 直接优化量子电路参数
- `qiskit.QuantumCircuit`: 构造量子电路
- `qiskit_aer.AerSimulator`: 用有限 shots 采样模拟量子测量

代码还会创建 `artifacts/` 文件夹，用于保存训练后的参数和配置。

这里还定义了量子线路运行的次数。

量子电路测量一次，只会得到一串 0/1 结果。为了估计概率，需要把同一个电路重复运行很多次。这里设置：
`SHOTS = 1024`
也就是每个句子对应的电路都会测量 1024 次。某个结果出现得越频繁，说明它对应的概率越高。

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

SEED = 7
SHOTS = 1024
np.random.seed(SEED)

SIMULATOR = AerSimulator(seed_simulator=SEED)

ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)

print('ready')
print(f'每个电路的测量次数 shots = {SHOTS}')

## 2. 数据集

这里使用非常小的三词句子数据集，每个句子都遵循主-谓-宾结构：

`subject verb object`

例如：

i love music\
friends dislike noise

标签含义：

- `1` 表示 positive
- `0` 表示 negative

这个数据集保持很小，方便观察量子电路训练的完整过程。测试句子只使用训练集中已经出现过的词，但会重新组合这些词，用来检查模型是否学到了动词中的情感倾向。

In [ ]:
train_data = [
    ('i love music', 1),
    ('people love games', 1),
    ('children enjoy books', 1),
    ('students like music', 1),
    ('friends enjoy games', 1),
    ('families love books', 1),
    ('i hate noise', 0),
    ('people dislike exams', 0),
    ('children hate homework', 0),
    ('students dislike noise', 0),
    ('friends hate exams', 0),
    ('families dislike homework', 0),
]

test_data = [
    ('i enjoy music', 1),
    ('students love books', 1),
    ('friends dislike noise', 0),
    ('families hate exams', 0),
]

# vocab: 所有句子中出现的单词集合
vocab = sorted({word for sentence, _ in train_data + test_data for word in sentence.split()})
word_to_id = {word: i for i, word in enumerate(vocab)} # word_to_id: 将单词映射到其在词汇表中的索引

# trainable_words: 训练集中第二个单词组成的集合
trainable_words = sorted({sentence.split()[1] for sentence, _ in train_data})
trainable_word_to_id = {word: i for i, word in enumerate(trainable_words)}  # trainable_word_to_id: 将训练集中第二个单词映射到其在训练集中的索引

print('vocab:', vocab)
print('trainable verb words:', trainable_words)

## 3. 构造句子量子电路

这一节定义句子如何进入量子电路。

电路使用 4 个 qubit：

- `q0`：主语
- `q1`：动词的一部分表示
- `q2`：动词的另一部分表示
- `q3`：宾语

每个词使用三个旋转门编码：

```text
RX(theta_0) -> RZ(theta_1) -> RX(theta_2)
```

可以把这三个角度看成这个词在量子电路里的“坐标”。不同词的角度不同，进入电路后形成的量子状态也不同。

动词会同时作用在 `q1` 和 `q2` 上，因为在这个小模型里，动词承担主要的情感信息。主语、动词、宾语编码完成后，电路使用三条 CNOT 门把四个 qubit 串联起来，让主语、动词、宾语之间产生关联。

```text
CX(0, 1)
CX(1, 2)
CX(2, 3)
```

可以把这一步理解为“把三个词合成一句话”的量子线路版本。

In [ ]:
# 定义一个函数，接受参数 params 并返回一个量子电路
def word_quantum_circuit(params):  
    qc = QuantumCircuit(1)
    qc.rx(float(params[0]), 0)
    qc.rz(float(params[1]), 0)
    qc.rx(float(params[2]), 0)
    return qc

# 初始化每个单词的参数，使用均匀分布在 [-0.2, 0.2] 范围内随机生成
base_word_params = np.random.default_rng(SEED).uniform(
    low=-0.2,
    high=0.2,
    size=(len(vocab), 3),  # 每个单词有 3 个参数
)

# 构建单词参数的函数，将训练集中第二个单词的参数从 theta 中提取出来，并更新 base_word_params
def build_word_params(theta):
    params = base_word_params.copy()  # 复制基础单词参数
    for word, idx in trainable_word_to_id.items():   # 遍历训练集中第二个单词及其索引
        params[word_to_id[word]] = theta[3 * idx:3 * idx + 3]  # 将 theta 中对应的参数更新到 params 中
    return params 

# 定义一个函数，接受一个句子和参数 theta，返回一个量子电路
def sentence_quantum_circuit(sentence, theta):
    # 将句子转换为小写并拆分为单词列表
    words = sentence.lower().split()
    # 检查句子是否为 3 个单词的 SVO 结构
    if len(words) != 3:
        raise ValueError(f'Only 3-word SVO sentences are supported, got: {sentence!r}')

    # 检查句子中的单词是否在词汇表中
    missing = [word for word in words if word not in word_to_id]
    if missing:
        raise KeyError(f'Unknown words: {missing}')

    # 构建单词参数
    word_params = build_word_params(theta)
    # 将句子中的单词映射为其在词汇表中的索引
    subj, verb, obj = [word_to_id[word] for word in words]

    qc = QuantumCircuit(4) # 创建一个包含 4 个量子比特的量子电路

    # 将主语的量子电路组合到第 0 个量子比特上
    qc.compose(word_quantum_circuit(word_params[subj]), qubits=[0], inplace=True)

    verb_params = word_params[verb] # 获取动词的参数
    qc.rx(float(verb_params[0]), 1) # 在第 1 个量子比特上应用 RX 门
    qc.rz(float(verb_params[1]), 1) # 在第 1 个量子比特上应用 RZ 门
    qc.rx(float(verb_params[2]), 1) # 在第 1 个量子比特上应用 RX 门
    
    qc.rx(float(verb_params[0]), 2) # 在第 2 个量子比特上应用 RX 门
    qc.rz(float(verb_params[1]), 2) # 在第 2 个量子比特上应用 RZ 门
    qc.rx(float(verb_params[2]), 2) # 在第 2 个量子比特上应用 RX 门

    # 将宾语的量子电路组合到第 3 个量子比特上
    qc.compose(word_quantum_circuit(word_params[obj]), qubits=[3], inplace=True)

    qc.barrier()  # 添加一个屏障，分隔电路的不同部分
    qc.cx(0, 1) # 在第 0 个量子比特和第 1 个量子比特之间应用 CNOT 门
    qc.cx(1, 2) # 在第 1 个量子比特和第 2 个量子比特之间应用 CNOT 门
    qc.cx(2, 3) # 在第 2 个量子比特和第 3 个量子比特之间应用 CNOT 门

    return qc # 返回构建好的量子电路

## 4. 量子测量

电路运行后会被重复测量 1024 次。一次测量只能得到一串 0/1，例如四个 qubit 可能被测成 `0101`。重复很多次后，就可以统计某些结果出现的频率，并用频率近似概率。

我们把动词对应的量子比特作为情感读出位置，因为数据中的积极/消极主要由动词决定，例如 `love` / `enjoy` / `like` 与 `hate` / `dislike`。

读出公式是：

`positive_score = (freq(q1=1) + freq(q2=1)) / 2`

这里的 `freq(q1=1)` 表示：在 1024 次测量中，`q1` 被测成 1 的次数除以 1024。`q2` 同理。

如果 `positive_score >= 0.5`，预测为 positive；否则预测为 negative。

这个读出过程不经过经典神经网络。训练和推理都直接使用量子电路测量频率得到的情感分数。

In [ ]:
# 定义一个函数，接受一个句子和参数 theta，返回一个测量后的量子电路
def measured_circuit(sentence, theta):
    qc = sentence_quantum_circuit(sentence, theta) # 创建一个量子电路
    qc.measure_all() # 对量子电路的所有量子比特进行测量
    return qc

# 定义一个函数，接受训练数据、参数 theta 和测量次数 shots，返回每个句子的正概率
def positive_probabilities(rows, theta, shots=SHOTS):
    # 为每个句子构建测量后的量子电路
    circuits = [measured_circuit(sentence, theta) for sentence, _ in rows]
    # 运行量子电路并获取结果
    result = SIMULATOR.run(circuits, shots=shots).result()

    probabilities = []  
    for circuit_index in range(len(circuits)):  # 遍历每个电路的索引
        counts = result.get_counts(circuit_index) # 获取每个电路的测量结果计数
        # 为啥要统计第 1 个量子比特和第 2 个量子比特为 1 的次数呢？
        # 因为在这个量子电路中，第 1、2 个量子比特的测量结果可以用来判断句子的正概率。
        # 通过统计这两个量子比特为 1 的次数，我们可以计算出它们的平均概率，从而得到句子的正概率。
        # 正概率意味着句子被认为是积极的或正面的。
        q1_ones = 0  # 统计第 1 个量子比特为 1 的次数
        q2_ones = 0  # 统计第 2 个量子比特为 1 的次数
        
        # 遍历测量结果计数，统计第 1 个量子比特和第 2 个量子比特为 1 的次数
        for bitstring, count in counts.items():
            bits = bitstring.replace(' ', '')  # 移除位字符串中的空格
            # Qiskit count strings are displayed as c3c2c1c0. 
            # Because measure_all maps q0->c0, q1->c1, q2->c2, q3->c3:
            # q1 is bits[-2], q2 is bits[-3].
            if bits[-2] == '1':  # 统计第 1 个量子比特为 1 的次数
                q1_ones += count
            if bits[-3] == '1':  # 统计第 2 个量子比特为 1 的次数
                q2_ones += count
                
        p_q1 = q1_ones / shots # 计算第 1 个量子比特为 1 的概率
        p_q2 = q2_ones / shots
        probabilities.append(float((p_q1 + p_q2) / 2.0))  # 计算第 1 个量子比特和第 2 个量子比特为 1 的平均概率，并将其添加到 probabilities 列表中
        
    return probabilities

In [ ]:
# 定义一个函数，接受一个句子、参数 theta 和测量次数 shots，返回该句子的正概率
def positive_probability(sentence, theta, shots=SHOTS):
    return positive_probabilities([(sentence, 0)], theta, shots=shots)[0]

# 定义一个函数，接受参数 theta 和训练数据 rows，返回二元交叉熵损失
def binary_cross_entropy(theta, rows):
    eps = 1e-7
    probs = positive_probabilities(rows, theta)
    losses = []
    for p, (_, label) in zip(probs, rows):  # 遍历每个句子的正概率和标签
        p = np.clip(p, eps, 1.0 - eps)  # 将概率限制在 [eps, 1.0 - eps] 范围内，避免对数运算的奇点
        # 计算二元交叉熵损失，并将其添加到 losses 列表中，是为了计算模型在训练数据上的平均损失，从而评估模型的性能。
        losses.append(-(label * np.log(p) + (1 - label) * np.log(1 - p)))
    return float(np.mean(losses))

# 定义一个函数，接受训练数据 rows 和参数 theta，返回准确率
def accuracy(rows, theta):
    probs = positive_probabilities(rows, theta)
    correct = 0
    for p, (_, label) in zip(probs, rows):  # 遍历每个句子的正概率和标签
        pred = int(p >= 0.5)  # 将正概率转换为二进制预测标签（0 或 1），如果正概率大于等于 0.5，则预测为 1，否则预测为 0
        correct += int(pred == label)  # 统计预测正确的样本数量
    return correct / len(rows)

## 5. 训练电路参数

训练目标是让 positive 句子的 `positive_score` 尽量接近 1，让 negative 句子的 `positive_score` 尽量接近 0。

使用的损失函数是二分类交叉熵：

`loss = - y log(p) - (1-y) log(1-p)`

其中：
- `y` 是真实标签
- `p` 是电路输出的 positive probability

优化器使用 `COBYLA`。它是一种不需要梯度的优化方法，适合这种“小规模量子电路 + 有限 shots 采样”的教学实验。

为了减少随机初始化带来的不稳定性，代码会尝试 3 个不同初始点，然后选择 loss 最低的一组参数作为最终电路参数。

In [ ]:
num_params = len(trainable_words) * 3  # 每个训练集中第二个单词有 3 个参数，因此总参数数量为 len(trainable_words) * 3

best_result = None
history = []

for seed in range(3):
    rng = np.random.default_rng(seed)
    theta0 = rng.uniform(-np.pi, np.pi, size=num_params)
    
    local_history = []
    
    def objective(theta):  # 定义一个目标函数，接受参数 theta 并返回二元交叉熵损失
        value = binary_cross_entropy(theta, train_data)
        local_history.append(value)
        return value

    # 使用 COBYLA 方法最小化目标函数，寻找最佳参数 theta    
    result = minimize(
        objective, # 目标函数
        theta0,    # 初始参数
        method='COBYLA',
        options={'maxiter': 120, 'rhobeg': 1.0, 'tol': 1e-4},  # 优化选项，rhobeg 是初始步长，tol 是收敛容差
    )
    
    # 记录每次优化的结果，包括种子、损失值、训练准确率和测试准确率
    print(
        f'seed={seed} loss={result.fun:.4f} '  # 打印当前种子和损失值
        f'train_acc={accuracy(train_data, result.x):.3f} '  # 打印训练准确率
        f'test_acc={accuracy(test_data, result.x):.3f}'  # 打印测试准确率
    )
    
    # 更新最佳结果和历史记录
    if best_result is None or result.fun < best_result.fun:
        best_result = result
        history = local_history

theta = best_result.x  # 获取最佳参数 theta

print('\nBest result')
print('loss:', round(binary_cross_entropy(theta, train_data), 4))
print('train accuracy:', round(accuracy(train_data, theta), 4))
print('test accuracy:', round(accuracy(test_data, theta), 4))

## 6. 训练曲线与测试结果

这一节展示两部分内容：
1. 优化过程中 loss 的变化
2. 测试句子的预测结果

每个测试样本会输出：
- 原始句子
- 真实标签
- 预测标签
- positive probability

这里的 `positive probability` 来自 1024 次测量的频率估计。由于测量采样带有随机性，重新运行时数值可能会有微小波动。

如果训练有效，positive 句子的概率应该大于 0.5，negative 句子的概率应该小于 0.5。

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history)
plt.xlabel('Objective evaluation')  # 目标评估
plt.ylabel('Binary cross entropy')  # 二元交叉熵损失
plt.title('Circuit Training Loss')  # 电路训练损失
plt.grid(True)
plt.show()

for sentence, label in test_data:
    p = positive_probability(sentence, theta)
    pred = int(p >= 0.5)
    label_name = 'positive' if pred == 1 else 'negative'
    # 打印句子、真实标签、预测标签、预测标签名称和正概率
    print(f'{sentence:26s} true={label} pred={pred} {label_name:8s} positive_prob={p:.4f}')

## 7. 展示训练后的量子电路

这一节选取一个测试句子，使用训练后的参数重新构造量子电路，并展示电路图。

图中的旋转门参数已经不是随机值，而是优化器训练后的结果。也就是说，展示出来的是已经学过情感分类任务的量子电路。

In [ ]:
sample_sentence = 'students love books'  # 选择一个样本句子进行测试
print('sample:', sample_sentence)
print('positive probability:', round(positive_probability(sample_sentence, theta), 4))

# 绘制样本句子的量子电路
sample_circuit = sentence_quantum_circuit(sample_sentence, theta)
# 显示量子电路图
sample_circuit.draw()